# 03_roster_optimizer.ipynb

Let's create a roster optimizer using linear programming. 

Start with just using data for a single season.

# Expected value of a player
Per the rules of the fantasy hockey league, skaters and goalies have different point systems.

## Skaters

The scoring for skaters (forwards and defencemen) is defined as:
- Goal: 2 Points
- Assist: 1 Point.


The expected value $v$ of a skater $i$ will be defined as:
$$
\mathbb{E}[v_i] = 2 \times \mathbb{E}[G_i] + 1 \times \mathbb{E}[A_i], 
$$

with

$$
\mathbb{E}[G_i] = \text{GPG}_i \times \mathbb{E}[\text{Games Played}],
$$

and 

$$
\mathbb{E}[A_i] = \text{APG}_i \times \mathbb{E}[\text{Games Played}],
$$

where $\mathbb{E}[G]$ is the expected goals, $\mathbb{E}[A]$ is the expected assists,  $\text{GPG}$ is the goals per game in the regular season, $\text{APG}$ is the assists per game in the regular season, and $\mathbb{E}[\text{Games Played}]$ is the expected number of games player $i$'s team will play in the playoffs.

## Goalies
The scoring for goalies is defined as:
- Win: 1 Point
- Assist: 1 Point
- Shutout: 2 Points


The expected value $v$ of a goalie $i$ will be defined as:

$$
\mathbb{E}[v_i] = 2 \times \mathbb{E}[\text{SO}_i] + 1 \times \mathbb{E}[W_i] + 1 \times \mathbb{E}[A_i], 
$$

with

$$
\mathbb{E}[\text{SO}_i] = \text{SOPG}_i \times \mathbb{E}[\text{Games Played}],
$$

and 

$$
\mathbb{E}[W_i] = \text{WPG}_i \times \mathbb{E}[\text{Games Played}],
$$

where $\mathbb{E}[\text{SO}]$ is the expected shutouts, $\mathbb{E}[W]$ is the expected wins,  $\text{SOPG}$ is the shutouts per game in the regular season, and $\text{WPG}$ is the wins per game in the regular season.

# The LP set-up
The roster is selected once at the beginning of the playoffs.

- 15 skaters total
  - 9 forwards
  - 6 defence
- 2 goalies

This presents a straightforward linear programming problem.

## Objective 
We will simplify the above notation and just call $v_i$ the expected value of player $i$.

Let $x_i=1$ if you pick player $i$, otherwise $x_i=0$. 

The objective function is then simply

$$
\text{max}\sum_i v_ix_i.
$$

## Constraints
The number of skaters in each position are our constraints, which can be written out as follows:

$$
\text{Total number of players}: \sum_i x_i = 17,
$$

$$
\text{Total number of forwards}: \sum_{i \in \text{forwards}} x_i = 9,
$$

$$
\text{Total number of defence}: \sum_{i \in \text{defence}} x_i = 6,
$$

$$
\text{Total number of goalies}: \sum_{i \in \text{goalies}} x_i = 2.
$$

We can also add strategic constraints. 

To limit the number of players we select per team as a way to hedge our roster selections (in case of a bracket upset for example):

$$
\text{Limit per team}: \sum_{i \in \text{Team T}} x_i \leq M_{\text{T}} \quad \forall \quad \text{T},
$$

where $M_{\text{T}}$ is our defined maximum number of players for a given team, $\text{T}$.

We can also impose a constraint that since we only choose two goalies total, it is best to not select two goalies who play for this same team.

Similarly, we do not want to pick a goalie who is not the starting goalie for that team.

# The Baseline Plan

- For skaters, compute the goals and assists per game in the regular season. 
- For goalies, compute the wins and shutouts per game in the regular season.

We will make the naive assumption that these rates continue in the playoffs. This assumption is what makes this a baseline model. In future versions, the expected value of a player in the playoffs will be the output of a machine learning model. 

- Multiply the rates by an estimate of how many games the given team's expected number of games played in the playoffs to get the expected value per player.

For a baseline implementation, maybe I could just use a sportsbook's predictions on playoff outcomes to estimate what round each team will make in the playoffs and then make an assumption that each series lasts say 6 games.

Or for running this LP problem on previous seasons I can just use the actual number of games each team played.

With the expected value per player calculated, we can then use scipy to solve the LP problem.


# Load data

Let's try this for the 2025-2026 season


In [1]:
import pandas as pd

from nhl_pool.common import stats_file_path

In [34]:
# SKATERS REGULAR SEASON
params = {
    "season": 20252026,
    "season_type": "regular",
    "player_type": "skaters"
}
skaters_regular_raw = pd.read_csv(stats_file_path(params))

# GOALIES REGULAR SEASON
params = {
    "season": 20252026,
    "season_type": "regular",
    "player_type": "goalies"
}
goalies_regular_raw = pd.read_csv(stats_file_path(params))

# SKATERS PLAYOFFS
# params = {
#     "season": 20252026,
#     "season_type": "playoffs",
#     "player_type": "skaters"
# }
# skaters_playoffs_raw = pd.read_csv(stats_file_path(params))

# # GOALIES PLAYOFFS
# params = {
#     "season": 20252026,
    # "season_type": "playoffs",
    # "player_type": "goalies"
# }
# goalies_playoffs_raw = pd.read_csv(stats_file_path(params))

## Pre-processing

### Handling duplicated players
In the regular season files, a player might show up more than one if they were traded. Let's remedy that by collapsing such that a player only appears once.

In [35]:
print(skaters_regular_raw['playerId'].duplicated().sum())
print(goalies_regular_raw['playerId'].duplicated().sum())

0
0


In [36]:
from nhl_pool.processing.players import collapse_players

skaters_regular = collapse_players(skaters_regular_raw, collapse_type="skater")
goalies_regular = collapse_players(goalies_regular_raw, collapse_type="goalie")

# Sanity checks
print(skaters_regular['playerId'].duplicated().sum())
print(goalies_regular['playerId'].duplicated().sum())

0
0


### positionCodes for Forwads

The league playoff points makes no distinction between C, LW, and RW. So we need to map these positions to simply forward, "F"

In [37]:
from nhl_pool.processing.players import map_positions

skaters_regular = map_positions(skaters_regular)
# skaters_playoffs = map_positions(skaters_playoffs_raw)

skaters_regular["positionCode"].value_counts()
# skaters_playoffs["positionCode"].value_counts()

positionCode
F    608
D    324
Name: count, dtype: int64

### Dropping players who played less than X games.

We want to select players who will regularly play. If a player only played in one game and got a point or two in that game they would have a high expected value because we assume a per game basis.

In [38]:
min_gamesPlayed = 15

skaters_regular = skaters_regular[skaters_regular["gamesPlayed"] >= min_gamesPlayed]
goalies_regular = goalies_regular[goalies_regular["gamesPlayed"] >= min_gamesPlayed]


### Dropping unneeded columns
For skaters we only need goals, assists, gamesPlayed and ID columns

For goalies we only need wins, shutouts, assists, gamesPlayed and ID columns

In [39]:
goalies_regular.columns

Index(['playerId', 'firstName', 'lastName', 'teamAbbrev', 'season',
       'seasonType', 'positionCode', 'gamesPlayed', 'gamesStarted', 'wins',
       'losses', 'overtimeLosses', 'goalsAgainstAverage', 'savePercentage',
       'shotsAgainst', 'saves', 'goalsAgainst', 'shutouts', 'goals', 'assists',
       'points', 'penaltyMinutes', 'timeOnIce', 'sourceFile'],
      dtype='str')

In [40]:
skaters_columns_to_keep = ['playerId', 'firstName', 'lastName', 'positionCode','teamAbbrev', 'goals', 'assists', 'gamesPlayed']
goalies_columns_to_keep = ['playerId', 'firstName', 'lastName', 'positionCode','teamAbbrev', 'wins', 'shutouts', 'assists', 'gamesPlayed']


skaters_regular = skaters_regular[skaters_columns_to_keep]
goalies_regular = goalies_regular[goalies_columns_to_keep]

skaters_regular.head()

,playerId,firstName,lastName,positionCode,teamAbbrev,goals,assists,gamesPlayed
0,8473986,Alex,Killorn,F,ANA,15,18,82
1,8474590,John,Carlson,D,ANA,14,46,71
2,8475184,Chris,Kreider,F,ANA,22,28,75
3,8475462,Radko,Gudas,D,ANA,2,11,56
4,8475798,Mikael,Granlund,F,ANA,19,22,58


# Expected Value of Players
## Skaters
For skaters, we only need to compute the goals and assists per game

In [41]:
skaters_regular["goalsPerGame"] = skaters_regular["goals"] / skaters_regular["gamesPlayed"]
skaters_regular["assistsPerGame"] = skaters_regular["assists"] / skaters_regular["gamesPlayed"]
skaters_regular.head()

,playerId,firstName,lastName,positionCode,teamAbbrev,goals,assists,gamesPlayed,goalsPerGame,assistsPerGame
0,8473986,Alex,Killorn,F,ANA,15,18,82,0.182927,0.219512
1,8474590,John,Carlson,D,ANA,14,46,71,0.197183,0.647887
2,8475184,Chris,Kreider,F,ANA,22,28,75,0.293333,0.373333
3,8475462,Radko,Gudas,D,ANA,2,11,56,0.035714,0.196429
4,8475798,Mikael,Granlund,F,ANA,19,22,58,0.327586,0.379310


Now we can create the expected value of each skater per game, valuePerGame

In [42]:
skaters_regular["valuePerGame"] = 2*skaters_regular["goalsPerGame"] + skaters_regular["assistsPerGame"]

skaters_regular.sort_values(by="valuePerGame", ascending=False).head()

,playerId,firstName,lastName,positionCode,teamAbbrev,goals,assists,gamesPlayed,goalsPerGame,assistsPerGame,valuePerGame
735,8476453,Nikita,Kucherov,F,TBL,44,86,76,0.578947,1.131579,2.289474
303,8478402,Connor,McDavid,F,EDM,48,90,82,0.585366,1.097561,2.268293
214,8477492,Nathan,MacKinnon,F,COL,53,74,80,0.662500,0.925000,2.250000
300,8477934,Leon,Draisaitl,F,EDM,35,62,65,0.538462,0.953846,2.030769
701,8484801,Macklin,Celebrini,F,SJS,45,70,82,0.548780,0.853659,1.951220


## Goalies
For goalies, we only need to compute the wins, shutouts, and assists per game

In [43]:
goalies_regular["winsPerGame"] = goalies_regular["wins"] / goalies_regular["gamesPlayed"]
goalies_regular["shutoutsPerGame"] = goalies_regular["shutouts"] / goalies_regular["gamesPlayed"]
goalies_regular["assistsPerGame"] = goalies_regular["assists"] / goalies_regular["gamesPlayed"]
goalies_regular.head()

,playerId,firstName,lastName,positionCode,teamAbbrev,wins,shutouts,assists,gamesPlayed,winsPerGame,shutoutsPerGame,assistsPerGame
1,8478024,Ville,Husso,G,ANA,10,0,0,20,0.500000,0.000000,0.000000
2,8480843,Lukas,Dostal,G,ANA,30,0,1,56,0.535714,0.000000,0.017857
4,8476914,Joonas,Korpisalo,G,BOS,14,1,0,31,0.451613,0.032258,0.000000
6,8480280,Jeremy,Swayman,G,BOS,31,2,0,55,0.563636,0.036364,0.000000
7,8479312,Alex,Lyon,G,BUF,20,3,1,36,0.555556,0.083333,0.027778


Now we can create the expected value of each goalie per game, valuePerGame

In [44]:
goalies_regular["valuePerGame"] = 2*goalies_regular["shutoutsPerGame"] + goalies_regular["winsPerGame"] + goalies_regular["assistsPerGame"]

goalies_regular.sort_values(by="valuePerGame", ascending=False).head()

,playerId,firstName,lastName,positionCode,teamAbbrev,wins,shutouts,assists,gamesPlayed,winsPerGame,shutoutsPerGame,assistsPerGame,valuePerGame
13,8483548,Brandon,Bussi,G,CAR,31,2,1,39,0.794872,0.051282,0.025641,0.923077
22,8475809,Scott,Wedgewood,G,COL,31,4,1,45,0.688889,0.088889,0.022222,0.888889
26,8479979,Jake,Oettinger,G,DAL,35,4,1,54,0.648148,0.074074,0.018519,0.814815
48,8478009,Ilya,Sorokin,G,NYI,29,7,1,55,0.527273,0.127273,0.018182,0.800000
73,8480981,Joel,Hofer,G,STL,24,6,0,46,0.521739,0.130435,0.000000,0.782609


# Expected number of games played

For now, let's hardcode the 2025-2026 expected number of games played based on the results of the Monte Carlo playoff simulations, as explained in another notebook.

In [ ]:
expected_games_played = {
    "COL": 15.351145,
    "BUF": 13.337275,
    "CAR": 13.22037,
    "DAL": 11.80482,
    "EDM": 11.429745,
    "MTL": 11.11616,
    "UTA": 11.06145,
    "PHI": 10.313495,
    "VGK": 10.301155,
    "TBL": 10.083305,
    "OTT": 9.82666,
    "MIN": 9.52918,
    "PIT": 9.106675,
    "ANA": 8.7488,
    "BOS": 8.232215,
    "LAK": 6.29599
}

# Roster Optimization

Now we can move on to actually selecting our roster.

First things first, we need to create clean DataFrames of just the skaters and goalies who will play in the playoffs.

In the next implementation, the API calls method will can be extended to get the roster of the team. For now, let's just use the playoffs tables and get the player information/team information from there.

In [ ]:
playoff_teams = set(expected_games_played.keys())

playoff_teams

{'ANA',
 'BOS',
 'BUF',
 'CAR',
 'COL',
 'DAL',
 'EDM',
 'LAK',
 'MIN',
 'MTL',
 'OTT',
 'PHI',
 'PIT',
 'TBL',
 'UTA',
 'VGK'}

In [47]:
mask = skaters_regular["teamAbbrev"].isin(playoff_teams)
skaters_playoffs = skaters_regular[mask]

mask = goalies_regular["teamAbbrev"].isin(playoff_teams)
goalies_playoffs_raw = goalies_regular[mask]

In [48]:
goalies_playoffs_raw

,playerId,firstName,lastName,positionCode,teamAbbrev,wins,shutouts,assists,gamesPlayed,winsPerGame,shutoutsPerGame,assistsPerGame,valuePerGame
1,8478024,Ville,Husso,G,ANA,10,0,0,20,0.500000,0.000000,0.000000,0.500000
2,8480843,Lukas,Dostal,G,ANA,30,0,1,56,0.535714,0.000000,0.017857,0.553571
4,8476914,Joonas,Korpisalo,G,BOS,14,1,0,31,0.451613,0.032258,0.000000,0.516129
6,8480280,Jeremy,Swayman,G,BOS,31,2,0,55,0.563636,0.036364,0.000000,0.636364
7,8479312,Alex,Lyon,G,BUF,20,3,1,36,0.555556,0.083333,0.027778,0.750000
8,8480045,Ukko-Pekka,Luukkonen,G,BUF,22,1,1,35,0.628571,0.028571,0.028571,0.714286
9,8481551,Colten,Ellis,G,BUF,8,1,1,16,0.500000,0.062500,0.062500,0.687500
10,8475883,Frederik,Andersen,G,CAR,16,0,1,35,0.457143,0.000000,0.028571,0.485714
13,8483548,Brandon,Bussi,G,CAR,31,2,1,39,0.794872,0.051282,0.025641,0.923077
22,8475809,Scott,Wedgewood,G,COL,31,4,1,45,0.688889,0.088889,0.022222,0.888889


In [49]:
playoff_cols = ["playerId", "firstName", "lastName", "teamAbbrev", "positionCode"]
skaters_playoffs = skaters_playoffs[playoff_cols]
goalies_playoffs = goalies_playoffs_raw[playoff_cols]

skaters_playoffs.head()

,playerId,firstName,lastName,teamAbbrev,positionCode
0,8473986,Alex,Killorn,ANA,F
1,8474590,John,Carlson,ANA,D
2,8475184,Chris,Kreider,ANA,F
3,8475462,Radko,Gudas,ANA,D
4,8475798,Mikael,Granlund,ANA,F


In [51]:
goalies_playoffs.head()

,playerId,firstName,lastName,teamAbbrev,positionCode
1,8478024,Ville,Husso,ANA,G
2,8480843,Lukas,Dostal,ANA,G
4,8476914,Joonas,Korpisalo,BOS,G
6,8480280,Jeremy,Swayman,BOS,G
7,8479312,Alex,Lyon,BUF,G


### Merging valuePerGame

Now we can left join the playoffs lists to the regular season to get the valuePerGame column for just the playoffs players.

In [52]:
skaters_playoffs = skaters_playoffs.merge(skaters_regular[["playerId", "valuePerGame"]],
                                          on="playerId",
                                          how="left")
skaters_playoffs["valuePerGame"] = skaters_playoffs["valuePerGame"].fillna(0)
skaters_playoffs.sort_values(by='valuePerGame', ascending=False)

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame
309,8476453,Nikita,Kucherov,TBL,F,2.289474
151,8478402,Connor,McDavid,EDM,F,2.268293
99,8477492,Nathan,MacKinnon,COL,F,2.250000
148,8477934,Leon,Draisaitl,EDM,F,2.030769
104,8480039,Martin,Necas,COL,F,1.769231
...,...,...,...,...,...,...
338,8479293,Brandon,Tanev,UTA,F,0.053571
240,8477073,Kurtis,MacDermid,OTT,F,0.052632
350,8484386,Dmitri,Simashev,UTA,D,0.035714
275,8483460,David,Jiricek,PHI,D,0.000000


In [53]:
skaters_playoffs["teamAbbrev"].unique()

<StringArray>
['ANA', 'BOS', 'BUF', 'CAR', 'COL', 'DAL', 'EDM', 'LAK', 'MIN', 'MTL', 'OTT',
 'PHI', 'PIT', 'TBL', 'UTA', 'VGK']
Length: 16, dtype: str

In [54]:
goalies_playoffs = goalies_playoffs.merge(goalies_regular[["playerId", "valuePerGame"]],
                                          on="playerId",
                                          how="left")
goalies_playoffs["valuePerGame"] = goalies_playoffs["valuePerGame"].fillna(0)
goalies_playoffs.sort_values(by='valuePerGame', ascending=False)

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame
8,8483548,Brandon,Bussi,CAR,G,0.923077
9,8475809,Scott,Wedgewood,COL,G,0.888889
12,8479979,Jake,Oettinger,DAL,G,0.814815
29,8476883,Andrei,Vasilevskiy,TBL,G,0.775862
19,8482661,Jesper,Wallstedt,MIN,G,0.771429
18,8479406,Filip,Gustavsson,MIN,G,0.760000
4,8479312,Alex,Lyon,BUF,G,0.750000
10,8478406,Mackenzie,Blackwood,COL,G,0.743590
23,8476999,Linus,Ullmark,OTT,G,0.714286
5,8480045,Ukko-Pekka,Luukkonen,BUF,G,0.714286


### Joining tables together
Now we want to put skaters and goalies into one table

In [55]:
players = pd.concat([skaters_playoffs, goalies_playoffs], ignore_index=True)
players

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame
0,8473986,Alex,Killorn,ANA,F,0.585366
1,8474590,John,Carlson,ANA,D,1.042254
2,8475184,Chris,Kreider,ANA,F,0.960000
3,8475462,Radko,Gudas,ANA,D,0.267857
4,8475798,Mikael,Granlund,ANA,F,1.034483
...,...,...,...,...,...,...
405,8477970,Vitek,Vanecek,UTA,G,0.363636
406,8478872,Karel,Vejmelka,UTA,G,0.687500
407,8478499,Adin,Hill,VGK,G,0.481481
408,8479394,Carter,Hart,VGK,G,0.611111


### Mapping expected games played by team and getting the total expected value of each player

In [ ]:
players["expectedGamesPlayed"] = players["teamAbbrev"].map(expected_games_played)
players

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame,expectedGamesPlayed
0,8473986,Alex,Killorn,ANA,F,0.585366,8.748800
1,8474590,John,Carlson,ANA,D,1.042254,8.748800
2,8475184,Chris,Kreider,ANA,F,0.960000,8.748800
3,8475462,Radko,Gudas,ANA,D,0.267857,8.748800
4,8475798,Mikael,Granlund,ANA,F,1.034483,8.748800
...,...,...,...,...,...,...,...
405,8477970,Vitek,Vanecek,UTA,G,0.363636,11.061450
406,8478872,Karel,Vejmelka,UTA,G,0.687500,11.061450
407,8478499,Adin,Hill,VGK,G,0.481481,10.301155
408,8479394,Carter,Hart,VGK,G,0.611111,10.301155


In [57]:
players["expectedValue"] = players["valuePerGame"] * players["expectedGamesPlayed"]

players.sort_values(by="expectedValue", ascending=False)

,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame,expectedGamesPlayed,expectedValue
99,8477492,Nathan,MacKinnon,COL,F,2.250000,15.351145,34.540076
104,8480039,Martin,Necas,COL,F,1.769231,15.351145,27.159718
151,8478402,Connor,McDavid,EDM,F,2.268293,11.429745,25.926007
148,8477934,Leon,Draisaitl,EDM,F,2.030769,11.429745,23.211174
309,8476453,Nikita,Kucherov,TBL,F,2.289474,10.083305,23.085461
...,...,...,...,...,...,...,...,...
338,8479293,Brandon,Tanev,UTA,F,0.053571,11.061450,0.592578
240,8477073,Kurtis,MacDermid,OTT,F,0.052632,9.826660,0.517193
350,8484386,Dmitri,Simashev,UTA,D,0.035714,11.061450,0.395052
175,8479421,Jacob,Moverare,LAK,D,0.000000,6.295990,0.000000


# Optimization via Linear Programming (Baseline)

We can solve this constrained optimization problem using Scipy.

Let's extract the columns we need

In [58]:
ev = players["expectedValue"].to_numpy(dtype=float)
pos = players["positionCode"]
team = players["teamAbbrev"]
N = len(players)

### Objective function
Maximising the expected value times the decision variable is the objective function.

Scipy's ```milp``` is a minimiser, so simply convert this to a minimisation problem.

In [59]:
c = -ev

### Indicator vectors for each position type
This is how we are going to force the position contraints

In [60]:
import numpy as np
is_forward = np.array(pos == "F").astype(int)
is_defence = np.array(pos == "D").astype(int)
is_goalie = np.array(pos == "G").astype(int)

## Build constraints as a matrix
Think about the constraints as being implemented as 
$$
Ax = b
$$

where $x$ is the decision variable (1 if select player else 0), $A$ are all the indicator vectors, and $b$ are the optimization constraints.

In [61]:
A_eq = np.vstack([np.ones_like(is_forward), is_forward, is_defence, is_goalie])

b_eq = np.array([17, 9, 6, 2])

print(A_eq.shape)
print(b_eq.shape)

(4, 410)
(4,)


## Solving with milp

In [62]:
from scipy.optimize import milp, Bounds, LinearConstraint

bounds = Bounds(lb=np.zeros(N), ub=np.ones(N))
integrality = np.ones(N, dtype=int)

# Equality is enforced with lb=ub
constraints_roster = LinearConstraint(A_eq, lb=b_eq, ub=b_eq) 

result = milp(c=c, constraints=constraints_roster, integrality=integrality, bounds=bounds)

In [63]:
roster_mask = result.x == 1

### Investigate the "optimal" roster

In [64]:
roster = players[roster_mask].sort_values(by='teamAbbrev')
print(f"Total Expected Value = {roster["expectedValue"].sum():.4f}")
roster

Total Expected Value = 330.5210


,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame,expectedGamesPlayed,expectedValue
52,8479420,Tage,Thompson,BUF,F,1.493827,13.337275,19.923584
57,8480839,Rasmus,Dahlin,BUF,D,1.207792,13.337275,16.108657
74,8476906,Shayne,Gostisbehere,CAR,D,1.145455,13.220370,15.143333
382,8483548,Brandon,Bussi,CAR,G,0.923077,13.220370,12.203418
105,8480069,Cale,Makar,COL,D,1.320000,15.351145,20.263511
383,8475809,Scott,Wedgewood,COL,G,0.888889,15.351145,13.645462
99,8477492,Nathan,MacKinnon,COL,F,2.250000,15.351145,34.540076
104,8480039,Martin,Necas,COL,F,1.769231,15.351145,27.159718
128,8480027,Jason,Robertson,DAL,F,1.719512,11.804820,20.298532
135,8482740,Wyatt,Johnston,DAL,F,1.597561,11.804820,18.858920


# Optimization via Linear Programming (with additional constraints)

As a way to hedge our bets, and to intelligently optimize the roster we will introduce some extra constraints.

1. Only selecting a maximum number of players from each team
2. Only select one goalie per team
3. Only select starting goaltenders (to be implemented later, need to label starting goalies manually).

We'll start with maximum number of players for each team:

In [65]:
teams = sorted(players["teamAbbrev"].unique())
team = players["teamAbbrev"].to_numpy()

max_per_team = 8

A_team = np.vstack([(team == t).astype(int) for t in teams])
con_team_max = LinearConstraint(A_team, lb=-np.inf, ub=max_per_team)

Now only one goalie per team:

In [66]:
A_goalie_team = np.vstack([((team == t) & (pos == "G")).astype(int) for t in teams])
con_goalie_max = LinearConstraint(A_goalie_team, lb=-np.inf, ub=1)

## Optimize roster

Add it with our original constraints

In [71]:
result = milp(
    c=c,
    constraints=[constraints_roster, con_team_max, con_goalie_max],
    integrality=integrality,
    bounds=bounds
)

roster_mask = result.x == 1

roster = players[roster_mask].sort_values(by='teamAbbrev')
print(f"Total Expected Value = {roster["expectedValue"].sum():.4f}")
roster.sort_values(by="positionCode", ascending=False)

Total Expected Value = 330.5210


,playerId,firstName,lastName,teamAbbrev,positionCode,valuePerGame,expectedGamesPlayed,expectedValue
382,8483548,Brandon,Bussi,CAR,G,0.923077,13.220370,12.203418
383,8475809,Scott,Wedgewood,COL,G,0.888889,15.351145,13.645462
52,8479420,Tage,Thompson,BUF,F,1.493827,13.337275,19.923584
135,8482740,Wyatt,Johnston,DAL,F,1.597561,11.804820,18.858920
309,8476453,Nikita,Kucherov,TBL,F,2.289474,10.083305,23.085461
224,8481540,Cole,Caufield,MTL,F,1.716049,11.116160,19.075880
148,8477934,Leon,Draisaitl,EDM,F,2.030769,11.429745,23.211174
151,8478402,Connor,McDavid,EDM,F,2.268293,11.429745,25.926007
128,8480027,Jason,Robertson,DAL,F,1.719512,11.804820,20.298532
104,8480039,Martin,Necas,COL,F,1.769231,15.351145,27.159718


# Takeaway Notes and Future Work

## Expected number of games played

As we can see, the expected number of games played per team is an extremely important factor in deciding the optimal player.
In this baseline example we had the benefit of knowing exactly how many series a team played in the playoffs. When using this workflow for an actual prediction, we will only have a list of teams. Our playoff bracket simulation framework will have to be good for this entire system to work.


## The "True" optimal roster
It would be interesting to see given the true playoff stats, what was the optimal roster? Could compare and contrast to the prediction at the start of the playoffs.

## Roster strategy evaluation
Since the Monte Carlo framework already generates a large number of simulated playoff bracket outcomes, these simulations could be used to evaluate alternative roster construction strategies. Rather than producing a single optimized roster, multiple potential rosters could be created under different optimization constraints (for example, changing the limit of teams to choose from). Each candidate roster could then be evaluated across the distribution of simulated playoff outcomes, allowing the strategy that performs best on average to be identified. This effectively introduces a layer of evaluation to the system, allowing the selection of a roster based on its expected value and on its robustness across many plausible playoff scenarios.

Another extension would be to incorporate uncertainty in player scoring during the evaluation stage. Rather than assuming every player's fantasy points per game exactly match their expected value, each simulated playoff could assign players a modest performance multiplier drawn from an appropriate probability distribution. Candidate roster strategies could then be evaluated across uncertainty in both playoff progression and player performance, allowing more robust roster construction strategies to be identified. This could be a form of sensitivity analysis.